# 03 - Embeddings & Vector Store - How do we index the chunks?

## Objective

Generate vector embeddings for the document chunks and store them in a vector database.

The objective is to build a searchable semantic index that supports efficient document retrieval.

This notebook focuses on experimentation rather than production implementation.

---

## Questions

- Which embedding model should be used?
- How are document chunks converted into embeddings?
- How should embeddings be stored?
- Can relevant chunks be retrieved using semantic similarity?

---

## Success Criteria

By the end of this notebook:

- Embeddings are successfully generated.
- Chunks are stored in ChromaDB.
- Similar chunks can be retrieved using natural language queries.
- The embedding strategy has been validated.

---

## Notes

The selected embedding model and vector database are documented in the corresponding ADRs.

This notebook validates their implementation before moving to the RAG pipeline.

In [7]:
from pathlib import Path
import re
import os 

from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    Docx2txtLoader, PyMuPDFLoader, TextLoader,
)
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Load documents
Reuse ingestion code of a previous notebook. At this point I'm only experimenting, in production this would not be duplicated.

In [3]:
# project paths 
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw/knowledge_base"

LOADERS = {".pdf": PyMuPDFLoader, ".docx": Docx2txtLoader, ".md": TextLoader, ".txt": TextLoader}

documents = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_dir():
        continue
    loader_cls = LOADERS.get(path.suffix.lower())
    if loader_cls is None:
        print(f"Skipping unsupported file: {path.name}")
        continue
    loader = loader_cls(str(path))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents.")

Skipping unsupported file: .DS_Store
Skipping unsupported file: .DS_Store
Loaded 6 documents.


In [4]:
def clean_text(text):
    # Collapse single newlines (likely mid-sentence wraps) into spaces,
    # but preserve intentional paragraph breaks (blank lines).
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse 3+ newlines down to a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse repeated spaces/tabs.
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

## Chunk documents

Reuse the selected strategy.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Generated {len(chunks)} chunks.")

Generated 50 chunks.


## Initialize the embedding model

Explain that embeddings convert text into vectors that capture semantic meaning.

In [6]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

## Create the vector store

In [8]:
db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vector_store._collection.count()} vectors.")

Vectorstore created with 50 vectors.


In [9]:
# Investigate the vectors 
collection = vector_store._collection

sample_embedding = collection.get(limit=1, include=["embeddings"])['embeddings'][0]
dimensions = len(sample_embedding)
print(f"Each vector has {dimensions} dimensions.")

Each vector has 3072 dimensions.


## Run the first similarity search

In [10]:
query = "What machine learning projects has the candidate worked on?"

results = vector_store.similarity_search(query=query, k=3)

In [11]:
# Inspect the retrieved results
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)
    print(doc.page_content)
    print(doc.metadata)

Result 1
Led advanced Machine Learning analyses in retail, including store clustering for price tiering, transaction level insights for basket composition, substitutability analysis for assortment optimization, and product role analysis, delivering KPIs that directly guided commercial decision making and pricing strategy

APNA, Melbourne – AUS

Data Analyst, Mar 2021 – Sep 2021

Analysed survey data across skills, wellbeing, and career development to identify key challenges and workforce trends.
{'source': '/Users/davidr/git_projects/ai-career-assistant/data/raw/knowledge_base/cv/CV_DR_FS.docx'}
Result 2
Wrangling and ETL processes using R to create the datasets to train Machine Learning models

Machine Learning Proof of Concept models using PySpark in Azure Databricks

Statistical Analysis and classification models

INAERO, Bogota – COL

Business analyst, Jul 2015 – Jan 2019

Developed demand forecasts and analytics covering passenger volumes, flight activity, cargo operations, and lo

## Additional experiments
**Watch for programming languages and language (English/Spanish/German)**

In [12]:
queries = [
    "How many years of experience does the candidate have in AI and generative AI?",
    "What experience is there with experimentation?",
    "What programming languages are used?",
]

for query in queries:
    print(f"\nQuery: {query}")

    results = vector_store.similarity_search(query=query, k=2)

    # Inspect the retrieved results
    for i, doc in enumerate(results, start=1):
        print("=" * 80)
        print(f"Result {i}")
        print("=" * 80)
        print(doc.page_content)
        print(doc.metadata)
        # print(results[0].page_content[:300])


Query: How many years of experience does the candidate have in AI and generative AI?
Result 1
Principal Data Scientist with 5+ years of experience in consulting delivering advanced analytics, BI, and machine learning solutions in the industries of Financial Services and Retail. I complement my technical experience with 4 additional years in finance and business analysis roles. I have a strong focus on solving commercial problems end-to-end, from framing business questions with senior stakeholders to building ML models that solve real world problems. I bring advanced proficiency in
{'source': '/Users/davidr/git_projects/ai-career-assistant/data/raw/knowledge_base/cv/CV_DR_FS.docx'}
Result 2
I have led the development of AI initiatives, creating agentic solutions to solve real commercial problems. I have a strong drive to deliver practical solutions that add value to the business, instead of complex AI/ML models that are disconnected from the business. In addition to that, my experience

## Conclusion

### Decision

Use:

- OpenAI `text-embedding-3-large`
- ChromaDB
- Top-k retrieval

### Rationale

- High-quality semantic retrieval.
- Simple Python integration.
- Low operational complexity.
- Suitable for the current project scope.

### Next Step

Build the Retrieval-Augmented Generation (RAG) pipeline.